In [ ]:
# %env Q_ALCHEMY_API_KEY=<api_key>

# One molecule, one Quantum I/O workflow

Prepare the **20-qubit carbon monoxide (CO/STO-3G) CISD state**, acquire observables, and reconstruct a QTucker state using the public Q-Alchemy SDK.

This replaces the preparation, acquisition, and reconstruction stages of the three `qtucker-examples/ibm_experiment` notebooks with **one service request**. The included sparse target is the same normalized CO state generated by the first notebook (288 amplitudes). Chemistry is already computed; you do not need PySCF or private QTucker packages locally.

Choose `ideal`, `noisy`, or `qpu` below. Ideal is the default. The noisy case imports a saved IBM device model and requires no IBM account. The QPU case executes on the IBM backend you select.

**Requirements:** SDK 0.5.0 and deployed Quantum I/O PineXQ 0.8.0 backed by core 0.11.0. Install this SDK revision into your notebook kernel before those versions are published. Set `Q_ALCHEMY_API_KEY` (or `PINEXQ_API_KEY`) in your environment or a local `.env`. For QPU mode, also set `IBM_QUANTUM_TOKEN` and optionally `IBM_QUANTUM_INSTANCE`.

For a released SDK: `pip install "q-alchemy-sdk-py[examples]>=0.5.0"`. This extra supplies the public Qiskit, Aer, IBM runtime, and notebook dependencies. Run from the SDK root or its `examples` folder.

In [ ]:
import os
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from q_alchemy import (
    ExecutionPlan, IBMQuantumCredentials, MeasurementPlan,
    QuantumExperiment, QuantumIOParams, QuantumIOService, Runtime, State,
)

load_dotenv()  # Reads credentials without embedding them in the notebook.


## Choose the experiment

These defaults match the original preparation and fitting controls: iterative preparation with a 0.001 estimated loss budget, blocks of four qubits, rank one, four restarts, and 600 fitting epochs. Start with `ideal` to check the workflow, then change only `MODE`.

The original observable family is retained as training data. `INDEPENDENT_VALIDATION=True` adds a separate family for held-out checking; it changes the experiment and increases acquisition work. Increasing `RANK`, `EPOCHS`, or `RESTARTS` may improve fitting, at additional cost.

In [ ]:
MODE = "ideal"  # "ideal", "noisy", or "qpu"
IBM_BACKEND = ""  # Required in QPU mode: use a device available to your account.
SHOTS = 4096
BLOCK_SIZE = 4
RANK = 1
EPOCHS = 600
RESTARTS = 4
INDEPENDENT_VALIDATION = False
OUTPUT_DIR = Path("co_reconstruction_outputs")

if MODE not in {"ideal", "noisy", "qpu"}:
    raise ValueError("MODE must be ideal, noisy, or qpu")


## Load the same sparse molecular target

CO has ten spatial orbitals, represented by twenty spin-orbital qubits, with geometry `C 0 0 0; O 0 0 1.128` in Ångström. The included file retains the original notebook's determinant-to-index convention unchanged. Each index now refers to the corresponding integer basis state in the SDK; no additional bit reversal is applied.

Only the sparse amplitudes are loaded and transmitted. No `2**20` dense vector is created.

In [ ]:
data_dir = Path("data") if Path("data/co_sto3g_cisd.npz").exists() else Path("examples/data")
with np.load(data_dir / "co_sto3g_cisd.npz", allow_pickle=False) as saved:
    indices = saved["indices"].copy()
    amplitudes = saved["amplitudes"].copy()
    num_qubits = int(saved["num_qubits"])

assert num_qubits == 20
assert np.isclose(np.vdot(amplitudes, amplitudes).real, 1.0)
target = State.sparse(num_qubits=num_qubits, indices=indices, amplitudes=amplitudes)
print(f"CO/STO-3G: {num_qubits} qubits, {len(indices)} nonzero amplitudes")


## Select how observables are acquired

- **Ideal:** direct sparse simulation evaluates expectations without shot noise or measurement circuits. `SHOTS` is not sampled on this path.
- **Noisy:** import `FakeAlmadenV2`, IBM's historical 20-qubit device snapshot, then serialize its Aer noise model and connectivity. Aer runs on the service with finite shots, using the matrix-product-state method to avoid a dense density matrix. This is a historical model, not current live calibration; noisy 20-qubit acquisition can still take substantial time.
- **QPU:** supply your IBM credentials separately and select a backend by name. Physical compilation and observable layout mapping are handled by Quantum I/O. Running the submission cell in this mode submits real hardware work.

The SDK sends only portable settings; it does not select preparation candidates or fit the model locally.

In [ ]:
credentials = None

if MODE == "ideal":
    acquisition = Runtime.qalchemy_sparse()

elif MODE == "noisy":
    from qiskit_ibm_runtime.fake_provider import FakeAlmadenV2
    from qiskit_aer.noise import NoiseModel

    device = FakeAlmadenV2()
    noise_model = NoiseModel.from_backend(device)
    assert device.num_qubits == num_qubits
    acquisition = Runtime.resource(
        "noisy-backend-simulator",
        provider="aer",
        backend="aer_simulator",
        backend_options={
            "method": "matrix_product_state",
            "noise_model": {
                "format": "qiskit-aer-noise-model-v1",
                "data": noise_model.to_dict(serializable=True),
            },
            "basis_gates": noise_model.basis_gates,
            "transpile": True,
            "transpile_options": {
                "basis_gates": noise_model.basis_gates,
                "coupling_map": [list(edge) for edge in device.coupling_map.get_edges()],
                "initial_layout": list(range(num_qubits)),
                "optimization_level": 1,
                "seed_transpiler": 123,
            },
            "estimator_options": {"seed_simulator": 123},
        },
    )

else:
    if not IBM_BACKEND.strip():
        raise ValueError("Set IBM_BACKEND to an accessible IBM device with at least 20 qubits")
    credentials = IBMQuantumCredentials(
        token=os.environ["IBM_QUANTUM_TOKEN"],
        instance=os.getenv("IBM_QUANTUM_INSTANCE") or None,
    )
    acquisition = Runtime.resource(
        "quantum-backend", provider="ibm", backend=IBM_BACKEND,
        execution_options={"transpile": True, "transpile_options": {"optimization_level": 1}},
    )

print(f"Acquisition mode: {MODE}")


## Describe preparation and reconstruction

There is no evolution circuit: we reconstruct the prepared molecular state, as in the original notebooks. Observable names, measurement circuits, acquisition, and fitting are generated on the service.

The fitted QTucker model stays implicit. Request amplitudes only on the input target's support and compare the full normalized model with that target. The exported subset is **not renormalized**, and model amplitudes outside the exported support are not assumed zero.

In [ ]:
experiment = QuantumExperiment(
    target=target,
    measurement_plan=MeasurementPlan.qtucker_reconstruction(
        validation=INDEPENDENT_VALIDATION,
        block_size=BLOCK_SIZE,
        rank=RANK,
        within_block_mode="all_paulis",
        cross_block_mode="all_pairs",
        rank_reduce_within_blocks=False,
        rank_reduce_cross_blocks=True,
        rank_observable_floor=1,
        rank_cross_block_floor=1,
        paulis="XYZ",
    ),
    metadata={"name": "co-sto3g-cisd-20-qubits", "acquisition_mode": MODE},
)
plan = ExecutionPlan(
    preparation_method="iterative_tucker",
    preparation_options={
        "basis_gates": ["u", "cx"],
        "fallback": False,
        "factors_size": 0,
        "max_fidelity_loss": 1e-3,
        "max_iterations": 1,
    },
    acquisition=acquisition,
    estimator=Runtime.qtucker(
        block_size=BLOCK_SIZE, rank=RANK, n_restarts=RESTARTS,
        fit_config={
            "optimizer": "adamw", "epochs": EPOCHS,
            "learning_rate": 0.03, "stderr_floor": 0.01,
            "loss_backend": "auto", "loss_compile_mode": "auto",
            "seed": 123, "verbose": False,
        },
    ),
    estimation_output={"support": "target", "target_fidelity": True},
    shots=SHOTS,
)


## Submit one job

This cell calls the deployed service in every mode. Preparation and fit settings are passed through to their owning libraries. Do not rerun it merely to redisplay results: the saved report below can be loaded independently.

In [ ]:
with QuantumIOService(
    QuantumIOParams(step_version="0.8.0"), ibm_credentials=credentials,
) as service:
    report = service.run(experiment, plan).result()

print(report.format_summary())


## Save and inspect the reconstruction

Inspect preparation `found`, its estimated loss, measured preflight fidelity (when available), and report warnings. The preparation loss estimate and reconstruction fidelity describe different stages. A failed search or poor reconstruction should not be treated as a successful molecular preparation.

`target_fidelity` compares the fitted normalized **pure-state surrogate** with the original CISD target. On noisy simulation or hardware it does not prove recovery of the full mixed state. With training-only data there is no independent held-out check, and matching these observables does not guarantee high target fidelity.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "report.json").write_text(report.to_json(indent=2), encoding="utf-8")
(OUTPUT_DIR / "experiment.json").write_text(experiment.to_json(indent=2), encoding="utf-8")
(OUTPUT_DIR / "execution_plan.json").write_text(plan.to_json(indent=2), encoding="utf-8")

if report.estimate is None or report.estimate.sparse_state is None:
    raise RuntimeError("No sparse reconstruction was returned; inspect the saved report and warnings")

estimate = report.estimate
estimate.sparse_state.save_npz(OUTPUT_DIR / "reconstructed.npz")
print("Target-to-estimated fidelity:", estimate.target_fidelity)
print("Exported amplitudes:", len(estimate.sparse_state.indices))
print("Probability on exported support:", estimate.sparse_state.metadata.get("returned_probability"))
print("Saved results to:", OUTPUT_DIR.resolve())


To inspect saved results without another service call:

```python
from q_alchemy import ExperimentReport
saved_report = ExperimentReport.from_json((OUTPUT_DIR / "report.json").read_text())
print(saved_report.format_summary())
```

The NPZ contains decimal basis indices, complex amplitudes, and JSON metadata, and loads with `np.load(path, allow_pickle=False)`. The report retains measured observables and fit diagnostics. See `examples/data/README.md` for target provenance.